[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/079.%20The%20Freight%20Carrier%20Selection%20%26%20Bidding%20Problem/P79-Tier-4_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 79. The Freight Carrier Selection and Bidding Problem
## Tier 4: Machine-Learning Augmentation

### Purpose and workflow
This notebook demonstrates how supervised machine-learning predictions can augment freight-carrier selection. It creates synthetic shipment records, engineers candidate features, trains models to predict carrier performance, scores carrier–lane options, and makes capacity-feasible assignments using a demand-ordered greedy rule. It then reports model diagnostics and an assumption-based business-impact calculation.

The workflow is: (1) define the carrier-selection data, (2) generate synthetic historical records, (3) engineer features, (4) train and evaluate prediction models, (5) score current carrier–lane options, (6) assign carriers subject to capacity, and (7) examine a conditional return-on-investment calculation.

### Scope and modeling assumptions
- The historical records are simulated, not observed carrier data. Model metrics therefore describe this synthetic data-generating process; they are not evidence of real-world predictive performance.
- The current decision considers three carriers and two lanes. Each lane is assigned to at most one carrier, and a carrier's total assigned demand cannot exceed its capacity.
- Bid prices are costs per demand unit. Reliability and service scores are percentages on a 0–100 scale.
- Estimated shipment weights and lane distances are scenario inputs. The synthetic data does not model dates, changing prices, uncertainty in capacity, contractual constraints, or rare disruptions.
- The assignment procedure is a greedy heuristic, not a global optimization method. It processes lanes from highest to lowest demand and selects the available carrier with the highest machine-learning score. A feasible assignment is not guaranteed if the heuristic reaches a lane for which no carrier has remaining capacity.
- Cost, reliability, and service values used to report an assignment are the problem's bid prices and supplied scores. Machine-learning predictions guide the ranking; they do not replace those reported values.

### Score used for carrier–lane ranking
For carrier \(i\) and lane \(j\), the machine-learning score is

$$\
S_{ij}^{\mathrm{ML}} = \alpha\,\frac{1000}{\widehat{c}_{ij}} + \beta\,\frac{\widehat{r}_{ij}}{100} + \gamma\,\frac{\widehat{s}_{ij}}{100} + 0.1\,\widehat{p}_{ij},
$$

where \(\widehat{c}_{ij}\) is predicted cost, \(\widehat{r}_{ij}\) is predicted reliability, \(\widehat{s}_{ij}\) is predicted service quality, and \(\widehat{p}_{ij}\) is the predicted probability of on-time delivery. The weights are \(\alpha=0.6\), \(\beta=0.3\), and \(\gamma=0.1\). Higher scores are preferred. The reciprocal cost term rewards lower predicted cost; the factor 1000 scales that term. The additional on-time term is a separate score bonus, so the score is a ranking index rather than a normalized probability or a dollar-valued objective.

For comparison, the traditional score uses the supplied bid price, reliability score, and service score, but not the on-time bonus. The capacity-feasible assignment step uses the machine-learning score to choose among available options. It does not solve a joint optimization over all lanes.

### Interpreting the results
Regression mean absolute error (MAE) is the average absolute difference between predictions and held-out targets, expressed in the target's units. Classification accuracy is the fraction of held-out on-time labels predicted correctly. Feature importance describes how a fitted random forest used its input features; it is not a causal effect. The ROI calculation is conditional on the stated comparison baseline, recurring savings assumption, and implementation costs, and should not be interpreted as a measured business outcome.

In [ ]:
# Import the libraries used for data handling, supervised learning, and plots.
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Fixed seeds make the synthetic data and model fits reproducible in this notebook.
np.random.seed(42)
random.seed(42)

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Define the carrier-selection data
The arrays below represent the decision inputs. Rows of the two-dimensional price and service arrays correspond to carriers; columns correspond to lanes. The one-dimensional reliability and capacity arrays are indexed by carrier, while demand is indexed by lane. Keeping these index meanings explicit makes later scenario construction and capacity checks easier to verify.

In [ ]:
# Carrier and lane order defines the row and column order of the arrays below.
carriers = ["Carrier 1", "Carrier 2", "Carrier 3"]
lanes = ["Lane 1 (NYC-CHI)", "Lane 2 (LAX-DAL)"]

# Bid prices are costs per demand unit: rows are carriers and columns are lanes.
bid_prices = np.array(
    [
        [500, 450],
        [480, 470],
        [520, 440],
    ]
)

# Reliability is one score per carrier; service is one score per carrier–lane pair.
reliability_scores = np.array([95, 88, 92])
service_scores = np.array(
    [
        [90, 85],
        [88, 92],
        [87, 90],
    ]
)

demand = np.array([100, 80])
capacity_limits = np.array([150, 120, 200])

# The weights define the cost, reliability, and service contributions to the score.
alpha = 0.6
beta = 0.3
gamma = 0.1

assert bid_prices.shape == (len(carriers), len(lanes))
assert service_scores.shape == bid_prices.shape
assert reliability_scores.shape == (len(carriers),)
assert capacity_limits.shape == (len(carriers),)
assert demand.shape == (len(lanes),)

print("Problem data:")
print(f"Carriers: {carriers}")
print(f"Lanes: {lanes}")
print(f"Demand: {demand}")
print(f"Weight parameters (alpha, beta, gamma): ({alpha}, {beta}, {gamma})")

## 2. Generate synthetic historical records
A supervised-learning model needs examples containing both input features and outcomes to predict. This function simulates such records by drawing a carrier and lane, adding noise to carrier-level reliability and service characteristics, and generating shipment conditions and outcomes. Because these records are synthetic, they are useful for demonstrating the workflow but cannot establish how accurately a model would perform on actual operations.

In [ ]:
def generate_historical_data(n_samples, carriers, lanes, bid_prices):
    """Generate reproducible synthetic carrier-performance records."""
    carrier_base_chars = {
        "Carrier 1": {
            "base_reliability": 95,
            "base_service": 90,
            "cost_factor": 1.0,
            "fleet_size": 50,
        },
        "Carrier 2": {
            "base_reliability": 88,
            "base_service": 88,
            "cost_factor": 0.95,
            "fleet_size": 35,
        },
        "Carrier 3": {
            "base_reliability": 92,
            "base_service": 87,
            "cost_factor": 1.05,
            "fleet_size": 60,
        },
    }

    if set(carriers) != set(carrier_base_chars):
        raise ValueError("The carrier list must match the synthetic carrier profiles.")
    if len(lanes) != bid_prices.shape[1] or len(carriers) != bid_prices.shape[0]:
        raise ValueError("Carrier and lane labels must match the bid-price array dimensions.")

    records = []
    for _ in range(n_samples):
        carrier = random.choice(carriers)
        carrier_idx = carriers.index(carrier)
        lane_idx = random.randrange(len(lanes))
        lane = lanes[lane_idx]
        base_char = carrier_base_chars[carrier]

        reliability = np.clip(
            base_char["base_reliability"] + np.random.normal(0, 5), 70, 100
        )
        service_quality = np.clip(
            base_char["base_service"] + np.random.normal(0, 8), 70, 100
        )

        # Actual cost is generated in the same per-unit scale as the bid prices.
        base_cost = bid_prices[carrier_idx, lane_idx]
        actual_cost = base_cost * base_char["cost_factor"] * np.random.uniform(0.9, 1.1)

        shipment_weight = np.random.uniform(1000, 50000)
        distance = np.random.uniform(500, 3000)
        weather_impact = np.random.uniform(0.8, 1.2)

        on_time_delivery = int(reliability > np.random.uniform(70, 100))
        damage_incidents = np.random.poisson(0.1)
        satisfaction = (
            reliability * 0.4
            + service_quality * 0.4
            + (100 - damage_incidents * 20) * 0.2
            + np.random.normal(0, 10)
        )
        satisfaction = np.clip(satisfaction, 0, 100)

        records.append(
            {
                "carrier": carrier,
                "lane": lane,
                "reliability": reliability,
                "service_quality": service_quality,
                "actual_cost": actual_cost,
                "shipment_weight": shipment_weight,
                "distance": distance,
                "weather_impact": weather_impact,
                "on_time_delivery": on_time_delivery,
                "damage_incidents": damage_incidents,
                "customer_satisfaction": satisfaction,
                "fleet_size": base_char["fleet_size"],
            }
        )

    return pd.DataFrame(records)


historical_data = generate_historical_data(
    n_samples=1000,
    carriers=carriers,
    lanes=lanes,
    bid_prices=bid_prices,
)

print("Synthetic historical data summary:")
print(f"Total records: {len(historical_data)}")
print(f"Distinct carriers: {historical_data['carrier'].nunique()}")
print(f"Distinct lanes: {historical_data['lane'].nunique()}")
print("\nCarrier distribution:")
print(historical_data["carrier"].value_counts())
print("\nLane distribution:")
print(historical_data["lane"].value_counts())

## 3. Engineer features
Feature engineering converts raw columns into inputs that may help a model represent the problem. Categorical carrier and lane labels are encoded as integers, and several ratios are calculated for analysis. A crucial modeling distinction is that some ratios below are derived from the outcomes being predicted. For example, cost-per-mile uses actual cost, and reliability-to-service uses reliability. Including such target-derived quantities as predictors would leak information from the answer into the inputs. They are retained in the engineered table for demonstration and inspection, but the training function deliberately excludes them.

In [ ]:
def engineer_features(df):
    """Encode categories and create descriptive features for historical records."""
    carrier_features = df.copy()

    le_carrier = LabelEncoder()
    le_lane = LabelEncoder()
    carrier_features["carrier_encoded"] = le_carrier.fit_transform(
        carrier_features["carrier"]
    )
    carrier_features["lane_encoded"] = le_lane.fit_transform(
        carrier_features["lane"]
    )

    carrier_features["cost_per_mile"] = (
        carrier_features["actual_cost"] / carrier_features["distance"]
    )
    carrier_features["cost_per_weight"] = (
        carrier_features["actual_cost"] / carrier_features["shipment_weight"]
    )
    carrier_features["weight_per_mile"] = (
        carrier_features["shipment_weight"] / carrier_features["distance"]
    )
    carrier_features["reliability_service_ratio"] = (
        carrier_features["reliability"] / carrier_features["service_quality"]
    )
    carrier_features["cost_reliability_ratio"] = (
        carrier_features["actual_cost"] / carrier_features["reliability"]
    )

    def categorize_risk(reliability):
        if reliability >= 95:
            return "Low"
        if reliability >= 85:
            return "Medium"
        return "High"

    carrier_features["risk_category"] = carrier_features["reliability"].apply(
        categorize_risk
    )
    le_risk = LabelEncoder()
    carrier_features["risk_category_encoded"] = le_risk.fit_transform(
        carrier_features["risk_category"]
    )

    return carrier_features, le_carrier, le_lane, le_risk


carrier_features, le_carrier, le_lane, le_risk = engineer_features(historical_data)

print("Engineered feature table:")
print(f"Shape: {carrier_features.shape}")
new_features = [
    "carrier_encoded",
    "lane_encoded",
    "cost_per_mile",
    "cost_per_weight",
    "weight_per_mile",
    "reliability_service_ratio",
    "cost_reliability_ratio",
    "risk_category",
    "risk_category_encoded",
]
for feature in new_features:
    print(f"  {feature}: {carrier_features[feature].dtype}")

print("\nRisk-category distribution:")
print(carrier_features["risk_category"].value_counts())

## 4. Train and evaluate prediction models
A random forest is an ensemble of decision trees: each tree makes a prediction, and the forest combines the trees' predictions. Regression forests predict continuous outcomes; a classification forest predicts a discrete label. Here, separate regression models predict reliability, service quality, and cost, while a classifier predicts whether a shipment is on time.

A held-out test set provides examples that are not used to fit each model. The feature list intentionally contains carrier, lane, shipment conditions, and fleet size, but not outcome-derived ratios or the target columns. This avoids direct target leakage. The data is synthetic, so a favorable test metric would still describe only the simulation.

In [ ]:
def train_ml_models(features_df):
    """Fit performance models and evaluate them on a held-out test set."""
    feature_cols = [
        "carrier_encoded",
        "lane_encoded",
        "shipment_weight",
        "distance",
        "weather_impact",
        "fleet_size",
    ]
    X = features_df[feature_cols]

    targets = {
        "reliability": features_df["reliability"],
        "service": features_df["service_quality"],
        "cost": features_df["actual_cost"],
        "ontime": features_df["on_time_delivery"],
    }

    # Identical fixed splits keep test rows aligned across all target models.
    X_train, X_test, train_indices, test_indices = train_test_split(
        X,
        np.arange(len(X)),
        test_size=0.2,
        random_state=42,
    )

    models = {}
    test_targets = {}
    for target_name, target_values in targets.items():
        y_train = target_values.iloc[train_indices]
        y_test = target_values.iloc[test_indices]
        test_targets[target_name] = y_test

        if target_name == "ontime":
            model = RandomForestClassifier(n_estimators=100, random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            metric_name = "acc"
            metric_value = accuracy_score(y_test, predictions)
        else:
            model = RandomForestRegressor(n_estimators=100, random_state=42)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)
            metric_name = "mae"
            metric_value = mean_absolute_error(y_test, predictions)

        models[target_name] = {
            "model": model,
            metric_name: metric_value,
            "predictions": predictions,
        }

    return models, feature_cols, X_test, test_targets


ml_models, feature_cols, X_test, test_targets = train_ml_models(carrier_features)

print("Held-out model performance:")
print(f"Reliability prediction MAE: {ml_models['reliability']['mae']:.3f} score points")
print(f"Service quality MAE: {ml_models['service']['mae']:.3f} score points")
print(f"Cost prediction MAE: {ml_models['cost']['mae']:.2f} cost units per demand unit")
print(f"On-time delivery accuracy: {ml_models['ontime']['acc']:.3f}")

rel_importance = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": ml_models["reliability"]["model"].feature_importances_,
    }
).sort_values("importance", ascending=False)

print("\nFeature importance for the reliability model:")
print(rel_importance.head(5).to_string(index=False))

## 5. Score current carrier–lane options
The fitted models are applied to one scenario for every carrier–lane pair. Estimated weight is demand multiplied by 100, and the lane distances are set to 1,000 and 1,500 miles; these are modeling inputs rather than observed measurements. The score combines predicted cost, reliability, service, and on-time probability according to the equation above. A random forest's predicted class is not a probability, so the on-time bonus uses the model's probability for class 1.

In [ ]:
def ml_enhanced_carrier_selection(
    ml_models,
    feature_cols,
    carriers,
    lanes,
    demand,
    bid_prices,
    reliability_scores,
    service_scores,
    alpha,
    beta,
    gamma,
    le_carrier,
    le_lane,
):
    """Predict and score each carrier–lane option using fitted models."""
    test_scenarios = []
    for carrier_idx, carrier in enumerate(carriers):
        for lane_idx, lane in enumerate(lanes):
            distance = 1000 if lane_idx == 0 else 1500
            shipment_weight = demand[lane_idx] * 100
            scenario = {
                "carrier": carrier,
                "lane": lane,
                "carrier_encoded": le_carrier.transform([carrier])[0],
                "lane_encoded": le_lane.transform([lane])[0],
                "shipment_weight": shipment_weight,
                "distance": distance,
                "weather_impact": 1.0,
                "fleet_size": [50, 35, 60][carrier_idx],
            }
            test_scenarios.append(scenario)

    test_df = pd.DataFrame(test_scenarios)
    X_scenarios = test_df[feature_cols]

    test_df["predicted_reliability"] = ml_models["reliability"]["model"].predict(
        X_scenarios
    )
    test_df["predicted_service"] = ml_models["service"]["model"].predict(
        X_scenarios
    )
    test_df["predicted_cost"] = ml_models["cost"]["model"].predict(X_scenarios)

    ontime_model = ml_models["ontime"]["model"]
    ontime_class_index = np.flatnonzero(ontime_model.classes_ == 1)
    if len(ontime_class_index) != 1:
        raise ValueError("The on-time classifier must be fitted with both binary classes.")
    test_df["predicted_ontime_prob"] = ontime_model.predict_proba(X_scenarios)[
        :, ontime_class_index[0]
    ]

    # The lower bound keeps the reciprocal cost term defined for any model output.
    predicted_cost = np.maximum(test_df["predicted_cost"].to_numpy(), 1e-9)
    test_df["ml_score"] = (
        alpha * (1000 / predicted_cost)
        + beta * test_df["predicted_reliability"] / 100
        + gamma * test_df["predicted_service"] / 100
        + 0.1 * test_df["predicted_ontime_prob"]
    )

    carrier_indices = test_df["carrier_encoded"].to_numpy(dtype=int)
    lane_indices = test_df["lane_encoded"].to_numpy(dtype=int)
    test_df["traditional_score"] = (
        alpha * (1000 / bid_prices[carrier_indices, lane_indices])
        + beta * reliability_scores[carrier_indices] / 100
        + gamma * service_scores[carrier_indices, lane_indices] / 100
    )
    return test_df


ml_results = ml_enhanced_carrier_selection(
    ml_models=ml_models,
    feature_cols=feature_cols,
    carriers=carriers,
    lanes=lanes,
    demand=demand,
    bid_prices=bid_prices,
    reliability_scores=reliability_scores,
    service_scores=service_scores,
    alpha=alpha,
    beta=beta,
    gamma=gamma,
    le_carrier=le_carrier,
    le_lane=le_lane,
)

print("Carrier–lane ranking scores:")
for lane in lanes:
    lane_results = ml_results[ml_results["lane"] == lane].sort_values(
        "ml_score", ascending=False
    )
    print(f"\n{lane}:")
    for _, row in lane_results.iterrows():
        print(
            f"  {row['carrier']}: ML score={row['ml_score']:.3f}, "
            f"traditional score={row['traditional_score']:.3f}"
        )
        print(
            f"    Predictions: reliability={row['predicted_reliability']:.1f}, "
            f"service={row['predicted_service']:.1f}, "
            f"cost={row['predicted_cost']:.2f}, "
            f"on-time probability={row['predicted_ontime_prob']:.2f}"
        )

## 6. Make capacity-feasible assignments
A greedy rule makes a sequence of local choices: process the lane with the larger demand first, then choose the highest-scoring carrier that can still accommodate that lane. Carrier utilization is the sum of the demands assigned so far. The capacity check is therefore \(u_i + d_j \leq K_i\), where \(u_i\) is current utilization, \(d_j\) is lane demand, and \(K_i\) is carrier capacity. Because each choice is made before later lanes are considered, the method is not guaranteed to find a globally best assignment.

In [ ]:
def make_ml_assignments(
    ml_results,
    carriers,
    lanes,
    demand,
    capacity_limits,
    bid_prices,
    reliability_scores,
    service_scores,
    le_carrier,
):
    """Assign lanes in descending demand order using ML scores and capacity checks."""
    assignments = [-1] * len(lanes)
    carrier_utilization = np.zeros(len(carriers), dtype=float)

    # Larger-demand lanes are handled first, as specified by the greedy procedure.
    lane_order = np.argsort(-demand)
    for lane_idx in lane_order:
        lane_name = lanes[lane_idx]
        lane_options = ml_results[ml_results["lane"] == lane_name]
        available_options = []

        for _, row in lane_options.iterrows():
            carrier_idx = int(le_carrier.transform([row["carrier"]])[0])
            if (
                carrier_utilization[carrier_idx] + demand[lane_idx]
                <= capacity_limits[carrier_idx]
            ):
                available_options.append((row, carrier_idx))

        if available_options:
            best_option, carrier_idx = max(
                available_options,
                key=lambda option: option[0]["ml_score"],
            )
            assignments[lane_idx] = carrier_idx
            carrier_utilization[carrier_idx] += demand[lane_idx]

    total_cost = 0.0
    total_reliability = 0.0
    total_service = 0.0
    assigned_lanes = 0
    for lane_idx, carrier_idx in enumerate(assignments):
        if carrier_idx != -1:
            total_cost += bid_prices[carrier_idx, lane_idx] * demand[lane_idx]
            total_reliability += reliability_scores[carrier_idx]
            total_service += service_scores[carrier_idx, lane_idx]
            assigned_lanes += 1

    avg_reliability = (
        total_reliability / assigned_lanes if assigned_lanes > 0 else 0.0
    )
    result = {
        "assignments": assignments,
        "carrier_utilization": carrier_utilization,
        "total_cost": total_cost,
        "avg_reliability": avg_reliability,
        "total_service": total_service,
        "assigned_lanes": assigned_lanes,
    }

    # These checks verify feasibility and consistency of the reported plan.
    assert np.all(carrier_utilization <= capacity_limits)
    assert result["assigned_lanes"] == sum(
        carrier_idx != -1 for carrier_idx in assignments
    )
    return result


ml_assignments = make_ml_assignments(
    ml_results=ml_results,
    carriers=carriers,
    lanes=lanes,
    demand=demand,
    capacity_limits=capacity_limits,
    bid_prices=bid_prices,
    reliability_scores=reliability_scores,
    service_scores=service_scores,
    le_carrier=le_carrier,
)

print("Greedy machine-learning-guided assignments:")
for lane_idx in np.argsort(-demand):
    carrier_idx = ml_assignments["assignments"][lane_idx]
    assignment = carriers[carrier_idx] if carrier_idx != -1 else "UNASSIGNED"
    print(f"{lanes[lane_idx]}: {assignment}")

print("\nAssignment summary:")
print(f"Total bid cost: ${ml_assignments['total_cost']:,.2f}")
print(f"Average assigned-carrier reliability: {ml_assignments['avg_reliability']:.1f}")
print(f"Total service score: {ml_assignments['total_service']:.1f}")
print(f"Assigned lanes: {ml_assignments['assigned_lanes']}/{len(lanes)}")
print(f"Carrier utilization: {ml_assignments['carrier_utilization']}")

## 7. Visualize model diagnostics and score rankings
The first panel shows test metrics with different target-specific units; their bar heights should not be compared as if they measured the same quantity. The second panel displays feature importance for the reliability model. The third compares machine-learning and traditional ranking scores for each carrier–lane option. The diagonal is a reference for equal scores, not a performance target.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Machine-Learning-Augmented Carrier Selection", fontsize=16, fontweight="bold")

# Model metrics have different units, so the axis is explicitly target-specific.
ax1 = axes[0, 0]
metric_names = ["Reliability MAE", "Service MAE", "Cost MAE", "On-time error"]
metric_values = [
    ml_models["reliability"]["mae"],
    ml_models["service"]["mae"],
    ml_models["cost"]["mae"],
    1 - ml_models["ontime"]["acc"],
]
metric_colors = ["lightblue", "lightgreen", "lightcoral", "lightyellow"]
bars = ax1.bar(metric_names, metric_values, color=metric_colors, alpha=0.85)
ax1.set_ylabel("Metric value (target-specific units)")
ax1.set_title("Held-Out Model Metrics")
ax1.tick_params(axis="x", rotation=25)
ax1.grid(True, axis="y", alpha=0.3)
for bar, metric_name, value in zip(bars, metric_names, metric_values):
    label = f"${value:.2f}" if metric_name == "Cost MAE" else f"{value:.3f}"
    ax1.annotate(
        label,
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        va="bottom",
    )

# Feature importance is model-specific and is not a causal effect.
ax2 = axes[0, 1]
importance_df = pd.DataFrame(
    {
        "feature": feature_cols,
        "importance": ml_models["reliability"]["model"].feature_importances_,
    }
).sort_values("importance", ascending=True)
ax2.barh(importance_df["feature"], importance_df["importance"], color="lightgreen")
ax2.set_xlabel("Random-forest feature importance")
ax2.set_title("Reliability Model Feature Importance")
ax2.grid(True, axis="x", alpha=0.3)

ax3 = axes[1, 0]
for carrier in carriers:
    carrier_data = ml_results[ml_results["carrier"] == carrier]
    ax3.scatter(
        carrier_data["traditional_score"],
        carrier_data["ml_score"],
        s=90,
        alpha=0.75,
        label=carrier,
    )
minimum_score = min(
    ml_results["traditional_score"].min(), ml_results["ml_score"].min()
)
maximum_score = max(
    ml_results["traditional_score"].max(), ml_results["ml_score"].max()
)
ax3.plot(
    [minimum_score, maximum_score],
    [minimum_score, maximum_score],
    "r--",
    alpha=0.6,
    label="Equal scores",
)
ax3.set_xlabel("Traditional score")
ax3.set_ylabel("Machine-learning score")
ax3.set_title("Ranking-Score Comparison")
ax3.legend()
ax3.grid(True, alpha=0.3)

# Display the assignment's bid cost as a single result, not as an unsupported
# comparison with results from other algorithms or data sets.
ax4 = axes[1, 1]
ax4.bar(
    ["ML-guided greedy assignment"],
    [ml_assignments["total_cost"]],
    color="lightblue",
    alpha=0.85,
)
ax4.set_ylabel("Total bid cost ($)")
ax4.set_title("Cost of the Reported Assignment")
ax4.tick_params(axis="x", rotation=15)
ax4.grid(True, axis="y", alpha=0.3)
ax4.annotate(
    f"${ml_assignments['total_cost']:,.2f}",
    (0, ml_assignments["total_cost"]),
    xytext=(0, 4),
    textcoords="offset points",
    ha="center",
    va="bottom",
)

plt.tight_layout()
plt.show()

## 8. Examine a conditional ROI scenario
Return on investment depends on the comparison baseline, the frequency of decisions, and the costs of developing and maintaining the system. The calculation below uses the supplied reference cost of 85,200 cost units per decision, assumes the same decision is made weekly, and retains the stated implementation and annual maintenance assumptions. These inputs are scenario assumptions, not measured savings. If the machine-learning-guided assignment costs more than the reference, the implied savings are negative and there is no positive payback under this simple model.

In [ ]:
def calculate_ml_roi(
    ml_assignments,
    greedy_reference_cost=85200,
    implementation_cost=50000,
    ongoing_annual_cost=10000,
    decisions_per_year=52,
):
    """Calculate scenario-based savings, first-year ROI, and simple payback."""
    if decisions_per_year <= 0:
        raise ValueError("decisions_per_year must be positive.")

    savings_per_decision = greedy_reference_cost - ml_assignments["total_cost"]
    annual_gross_savings = savings_per_decision * decisions_per_year
    first_year_net_benefit = (
        annual_gross_savings - implementation_cost - ongoing_annual_cost
    )
    total_first_year_cost = implementation_cost + ongoing_annual_cost
    first_year_roi = (
        first_year_net_benefit / total_first_year_cost * 100
        if total_first_year_cost > 0
        else float("nan")
    )

    net_savings_per_week = (
        savings_per_decision - ongoing_annual_cost / decisions_per_year
    )
    payback_period_weeks = (
        implementation_cost / net_savings_per_week
        if net_savings_per_week > 0
        else float("inf")
    )

    return {
        "savings_per_decision": savings_per_decision,
        "annual_gross_savings": annual_gross_savings,
        "first_year_net_benefit": first_year_net_benefit,
        "implementation_cost": implementation_cost,
        "ongoing_annual_cost": ongoing_annual_cost,
        "first_year_roi": first_year_roi,
        "payback_period_weeks": payback_period_weeks,
    }


roi_analysis = calculate_ml_roi(ml_assignments)

print("Conditional ROI scenario:")
print(f"Savings per decision versus reference: ${roi_analysis['savings_per_decision']:,.2f}")
print(f"Annual gross savings: ${roi_analysis['annual_gross_savings']:,.2f}")
print(f"Implementation cost: ${roi_analysis['implementation_cost']:,.2f}")
print(f"Annual maintenance cost: ${roi_analysis['ongoing_annual_cost']:,.2f}")
print(f"First-year net benefit: ${roi_analysis['first_year_net_benefit']:,.2f}")
print(f"First-year ROI: {roi_analysis['first_year_roi']:.1f}%")
if np.isfinite(roi_analysis['payback_period_weeks']):
    print(f"Simple payback period: {roi_analysis['payback_period_weeks']:.1f} weeks")
else:
    print("Simple payback period: no positive payback under these assumptions")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Conditional Business-Impact Scenario", fontsize=15, fontweight="bold")

financial_values = [
    roi_analysis["annual_gross_savings"],
    -roi_analysis["implementation_cost"],
    -roi_analysis["ongoing_annual_cost"],
    roi_analysis["first_year_net_benefit"],
]
financial_labels = [
    "Annual gross\nsavings",
    "Implementation\ncost",
    "Annual maintenance\ncost",
    "First-year net\nbenefit",
]
financial_colors = ["lightgreen", "lightcoral", "lightcoral", "lightblue"]
bars = ax1.bar(financial_labels, financial_values, color=financial_colors, alpha=0.85)
ax1.axhline(0, color="black", linewidth=0.8)
ax1.set_ylabel("Amount ($)")
ax1.set_title("First-Year Financial Components")
ax1.grid(True, axis="y", alpha=0.3)
for bar, value in zip(bars, financial_values):
    offset = 3 if value >= 0 else -3
    vertical_alignment = "bottom" if value >= 0 else "top"
    ax1.annotate(
        f"${value:,.0f}",
        (bar.get_x() + bar.get_width() / 2, value),
        xytext=(0, offset),
        textcoords="offset points",
        ha="center",
        va=vertical_alignment,
    )

weeks = np.arange(0, 53)
net_savings_per_week = (
    roi_analysis["savings_per_decision"]
    - roi_analysis["ongoing_annual_cost"] / 52
)
cumulative_net_savings = weeks * net_savings_per_week
ax2.plot(weeks, cumulative_net_savings, color="green", linewidth=2, label="Cumulative net savings")
ax2.axhline(
    roi_analysis["implementation_cost"],
    color="red",
    linestyle="--",
    linewidth=2,
    label="Implementation cost",
)
if np.isfinite(roi_analysis["payback_period_weeks"]):
    ax2.axvline(
        roi_analysis["payback_period_weeks"],
        color="orange",
        linestyle=":",
        linewidth=2,
        label=f"Payback ({roi_analysis['payback_period_weeks']:.1f} weeks)",
    )
ax2.set_xlabel("Weeks")
ax2.set_ylabel("Cumulative amount ($)")
ax2.set_title("Payback Scenario")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Strengths, limitations, and appropriate use
### Potential strengths
- Historical records can help quantify patterns that are difficult to encode directly in a hand-built score.
- Predictions can incorporate shipment conditions alongside carrier and lane identifiers.
- A score can combine predicted cost and service outcomes with an operational preference for on-time delivery.

### Limitations
- Predictions depend on representative, accurately recorded historical data. Synthetic training data cannot substitute for operational validation.
- Random-forest feature importance is descriptive, not causal, and a model may perform poorly when future conditions differ from training conditions.
- The score weights and on-time bonus are modeling choices. Their scale affects rankings, and they do not themselves establish a financially optimal trade-off.
- The greedy assignment is locally guided and can miss a better joint assignment or leave a lane unassigned.
- The scenario-based ROI is sensitive to its baseline, savings frequency, and cost assumptions; it is not a forecast without supporting evidence.

### When this approach may be useful
Machine-learning augmentation may be worth evaluating when an organization has sufficient, relevant shipment history and wants performance predictions to inform carrier rankings. Before operational use, assess performance on representative unseen data, monitor predictions over time, review the score's business meaning, and confirm that assignment and contractual constraints are adequately represented.

## 10. Final assessment
The final summary connects the trained models, score-guided assignment, capacity checks, and conditional financial scenario. Model metrics describe held-out synthetic records; assignment metrics use the supplied bid prices and carrier scores; ROI values depend on the stated reference assumptions. These quantities answer different questions and should be interpreted separately.

In [ ]:
print("=" * 72)
print("MACHINE-LEARNING-AUGMENTED CARRIER SELECTION: FINAL ASSESSMENT")
print("=" * 72)

print("\nAssignments:")
for lane_idx in np.argsort(-demand):
    carrier_idx = ml_assignments["assignments"][lane_idx]
    assignment = carriers[carrier_idx] if carrier_idx != -1 else "UNASSIGNED"
    print(f"  {lanes[lane_idx]}: {assignment}")

print("\nAssignment measures:")
print(f"  Total bid cost: ${ml_assignments['total_cost']:,.2f}")
print(f"  Average assigned-carrier reliability: {ml_assignments['avg_reliability']:.1f}")
print(f"  Total service score: {ml_assignments['total_service']:.1f}")
print(f"  Assigned lanes: {ml_assignments['assigned_lanes']}/{len(lanes)}")
print(f"  Capacity used by carrier: {ml_assignments['carrier_utilization']}")

print("\nHeld-out synthetic-data model metrics:")
print(f"  Reliability MAE: {ml_models['reliability']['mae']:.3f} score points")
print(f"  Service MAE: {ml_models['service']['mae']:.3f} score points")
print(f"  Cost MAE: {ml_models['cost']['mae']:.2f} cost units per demand unit")
print(f"  On-time accuracy: {ml_models['ontime']['acc']:.3f}")

print("\nConditional financial scenario:")
print(f"  Savings per decision versus reference: ${roi_analysis['savings_per_decision']:,.2f}")
print(f"  Annual gross savings: ${roi_analysis['annual_gross_savings']:,.2f}")
print(f"  First-year net benefit: ${roi_analysis['first_year_net_benefit']:,.2f}")
print(f"  First-year ROI: {roi_analysis['first_year_roi']:.1f}%")
if np.isfinite(roi_analysis['payback_period_weeks']):
    print(f"  Simple payback period: {roi_analysis['payback_period_weeks']:.1f} weeks")
else:
    print("  Simple payback period: no positive payback under these assumptions")

print("\nInterpretation:")
print("  The models provide predictions for the simulated data-generating process.")
print("  Predictions guide a capacity-feasible greedy assignment; they do not prove global optimality.")
print("  The financial scenario is conditional on its reference cost and cost assumptions.")
print("=" * 72)